In [28]:
import torch
from torchvision import datasets
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2


In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import v2


In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import v2


In [12]:
train_dataset = datasets.MNIST(
    root = 'root',
    train = True,
    download = True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [14]:
test_dataset = datasets.MNIST(
    root = 'root',
    train = False,
    download = True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [15]:
train_dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: root
    Split: Train
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

In [29]:
class Ds(Dataset):
  def __init__(self , images , labels):
    self.images = images
    self.labels = labels

  def __getitem__(self, index):
    image = self.images[index]
    label = self.labels[index]
    return image , label
  def __len__(self):
    return len(self.images)

In [17]:
image, label = train_dataset[0]

In [18]:
image

Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [20]:
label

5

In [24]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [25]:
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)


In [50]:
from torch import nn

class Neural(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.stack = nn.Sequential(
        nn.Linear(28*28, 784),
        nn.ReLU(),
        nn.Linear(784, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    )
  def forward(self , x):
    x = self.flatten(x)
    logit = self.stack(x)
    return logit

In [51]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Neural().to(device)
print(model)

Neural(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (stack): Sequential(
    (0): Linear(in_features=784, out_features=784, bias=True)
    (1): ReLU()
    (2): Linear(in_features=784, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [70]:
learning_rate = 1e-4|
batch_size = 64
epochs = 10

SyntaxError: invalid syntax (2418531735.py, line 1)

In [53]:
loss_fn = nn.CrossEntropyLoss()

In [54]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [59]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [71]:
epoches = 10

for t in range(epoches):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.145880  [   64/60000]
loss: 0.152043  [ 6464/60000]
loss: 0.284971  [12864/60000]
loss: 0.487623  [19264/60000]
loss: 0.313132  [25664/60000]
loss: 0.406849  [32064/60000]
loss: 0.163204  [38464/60000]
loss: 0.436268  [44864/60000]
loss: 0.335159  [51264/60000]


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1541/1948439836.py", line 5, in <cell line: 0>
    train_loop(train_dataloader, model, loss_fn, optimizer)
  File "/tmp/ipykernel_1541/2759824000.py", line 6, in train_loop
    for batch, (X, y) in enumerate(dataloader):
                         ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 801, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx


KeyboardInterrupt



In [63]:
image, label = test_dataset[0]

In [65]:
image = image.to(device)
prediction = model(image)

In [66]:
prediction.argmax()

tensor(7, device='cuda:0')

In [67]:
label

7